# 02 - Ön İşleme

Bu not defteri, eğitim öncesinde uyguladığımız veri hazırlık adımlarını şeffaf biçimde doğrular.

Akış:
1. 0 değerlerini gerekli alanlarda NaN'e dönüştürme
2. Medyan imputasyon
3. Standardizasyon
4. Yalnızca eğitim verisinde SMOTE


In [ ]:
# Ortamı hazırlıyor ve proje kökünü Python yoluna ekliyoruz.
from pathlib import Path
import sys

proje_kok = Path.cwd().resolve()
while proje_kok.name != 'diyabet_risk_tahmini' and proje_kok.parent != proje_kok:
    proje_kok = proje_kok.parent

if proje_kok.name != 'diyabet_risk_tahmini':
    raise RuntimeError('Notebook proje kökünden veya alt klasörlerinden çalıştırılmalıdır.')

if str(proje_kok) not in sys.path:
    sys.path.insert(0, str(proje_kok))

print(f'Proje kökü: {proje_kok}')


In [ ]:
# Ön işleme adımlarında kullanacağımız araçları içe aktarıyoruz.
import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

from makine_ogrenmesi.kaynak.veri_yukleyici import veri_setini_yukle
from makine_ogrenmesi.kaynak.on_isleme import (
    sifirlari_nan_yap,
    median_imputer_olustur,
    standard_scaler_olustur,
)
from makine_ogrenmesi.kaynak.ozellik_yapilandirmasi import (
    HEDEF_KOLONU,
    OZELLIK_KOLONLARI,
    SIFIRI_EKSIK_SAYILAN_KOLONLAR,
)


In [ ]:
# Veriyi yükleyip özellik ve hedef değişkenlerini ayırıyoruz.
veri = veri_setini_yukle(proje_kok / 'makine_ogrenmesi' / 'veri' / 'ham' / 'diabetes.csv')
X = veri[OZELLIK_KOLONLARI].copy()
y = veri[HEDEF_KOLONU].copy()

print('Orijinal boyut:', X.shape)
X.head()


In [ ]:
# Sıfır değerlerin NaN'e dönüşümünü önce-sonra karşılaştırmasıyla kontrol ediyoruz.
sifir_once = (X[SIFIRI_EKSIK_SAYILAN_KOLONLAR] == 0).sum().rename('sıfır_öncesi')
X_nan = sifirlari_nan_yap(X, SIFIRI_EKSIK_SAYILAN_KOLONLAR)
nan_sonra = X_nan[SIFIRI_EKSIK_SAYILAN_KOLONLAR].isna().sum().rename('nan_sonrası')

pd.concat([sifir_once, nan_sonra], axis=1)


In [ ]:
# Veriyi stratified olacak şekilde eğitim ve test olarak ayırıyoruz.
X_train, X_test, y_train, y_test = train_test_split(
    X_nan,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print('Eğitim boyutu:', X_train.shape)
print('Test boyutu  :', X_test.shape)
print('Eğitim sınıf dağılımı:')
print(y_train.value_counts().sort_index())


In [ ]:
# Medyan imputasyonu yalnızca eğitim verisine fit edip test verisine aynı dönüşümü uyguluyoruz.
imputer = median_imputer_olustur()
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=OZELLIK_KOLONLARI, index=X_train.index)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=OZELLIK_KOLONLARI, index=X_test.index)

print('Eğitimde kalan NaN sayısı:', int(X_train_imp.isna().sum().sum()))
print('Testte kalan NaN sayısı  :', int(X_test_imp.isna().sum().sum()))
X_train_imp.head()


In [ ]:
# Ölçekleme adımından sonra özelliklerin ortalama ve standart sapma değerlerini kontrol ediyoruz.
scaler = standard_scaler_olustur()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_imp), columns=OZELLIK_KOLONLARI, index=X_train_imp.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imp), columns=OZELLIK_KOLONLARI, index=X_test_imp.index)

istatistik = pd.DataFrame({
    'ortalama': X_train_scaled.mean(),
    'std': X_train_scaled.std(ddof=0),
})
istatistik


In [ ]:
# SMOTE işlemini yalnızca eğitim verisine uygulayıp sınıf dengesini gözlemliyoruz.
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)

sinif_ozet = pd.DataFrame({
    'öncesi': y_train.value_counts().sort_index(),
    'SMOTE_sonrası': pd.Series(y_train_bal).value_counts().sort_index(),
})
sinif_ozet.index = ['sınıf_0', 'sınıf_1']
sinif_ozet


## Kısa değerlendirme

- Veri sızıntısını engellemek için tüm dönüşümler eğitim verisi üstünden öğrenilmelidir.
- SMOTE test verisine uygulanmaz; aksi durumda değerlendirme yapay biçimde iyileşir.
- Bu temizlik adımları, model kıyasının adil olmasını sağlar.
